# **Truthenix - An AI-Powered Fake News Detection Platform**

**1. Import the libraries**

In [47]:
import numpy as np
import pandas as pd
import itertools

**2. Connect to the dataset**

In [48]:
df=pd.read_csv('/content/news.csv')

In [49]:
df.head()

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [50]:
df.shape

(6335, 4)

**3. Check for null value**

In [51]:
df.isnull().sum()

,0
Unnamed: 0,0
title,0
text,0
label,0


In [52]:
df.dropna(inplace=True)

In [53]:
print(df.shape)
df.isnull().sum()

(6335, 4)


,0
Unnamed: 0,0
title,0
text,0
label,0


**4. Examine the target variable - label**

In [54]:
print(df['label'].value_counts())
df.head()

label
REAL    3171
FAKE    3164
Name: count, dtype: int64


,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


**5. Combine the title and text columns for better performance**

In [55]:
df['text'] = df['title'] + ' ' + df['text']
df.head()

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,You Can Smell Hillary’s Fear Daniel Greenfield...,FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Watch The Exact Moment Paul Ryan Committed Pol...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,Kerry to go to Paris in gesture of sympathy U....,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,Bernie supporters on Twitter erupt in anger ag...,FAKE
4,875,The Battle of New York: Why This Primary Matters,The Battle of New York: Why This Primary Matte...,REAL


**6. Perform Data Cleaning operations to removing unnessesary punctuations and also removing stopwords.**

In [56]:
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import nltk
nltk.download('stopwords')

ps = PorterStemmer()
all_stopwords = stopwords.words('english')
all_stopwords.remove('not')

def preprocess_text(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.split()
    text = [ps.stem(word) for word in text if not word in set(all_stopwords)]
    text = ' '.join(text)
    return text

df['text'] = df['text'].apply(preprocess_text)
df.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,smell hillari fear daniel greenfield shillman ...,FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,watch exact moment paul ryan commit polit suic...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,kerri go pari gestur sympathi u secretari stat...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,berni support twitter erupt anger dnc tri warn...,FAKE
4,875,The Battle of New York: Why This Primary Matters,battl new york primari matter primari day new ...,REAL


**7. Prepare data for machine learning - Convert texts into numerical format**

In [57]:
X = df['text']
y = df['label']

# Convert 'label' to numerical format (e.g., FAKE: 0, REAL: 1)
y = y.map({'FAKE': 0, 'REAL': 1})

print("First 5 entries of features (X):\n", X.head())
print("\nFirst 5 entries of target (y):\n", y.head())

First 5 entries of features (X):
 0    smell hillari fear daniel greenfield shillman ...
1    watch exact moment paul ryan commit polit suic...
2    kerri go pari gestur sympathi u secretari stat...
3    berni support twitter erupt anger dnc tri warn...
4    battl new york primari matter primari day new ...
Name: text, dtype: object

First 5 entries of target (y):
 0    0
1    0
2    1
3    0
4    1
Name: label, dtype: int64


**8. Split the dataset into training and testing part**

In [58]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**9. Use TfidVectorizer to convert data into numerical form and fed the data into model**

In [59]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)

tfidf_train = tfidf_vectorizer.fit_transform(X_train)
tfidf_test = tfidf_vectorizer.transform(X_test)

**10. Use PassiveAggressiveClassifier as it will be the best option for this project**

In [60]:
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# Initialize a PassiveAggressiveClassifier
pac = PassiveAggressiveClassifier(max_iter=50, random_state=42)

# Fit the model on training data
pac.fit(tfidf_train, y_train)

# Predict on the test set
y_pred = pac.predict(tfidf_test)

# Calculate accuracy and display confusion matrix
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f'Accuracy: {accuracy:.4f}')
print('\nConfusion Matrix:\n', conf_matrix)

Accuracy: 0.9313

Confusion Matrix:
 [[583  45]
 [ 42 597]]


**11. So the accuracy of the model is ~93.13% with an impressive confusion matrix.**

**Project Conclusion:**

This project successfully developed and evaluated a machine learning model for fake news detection. Key steps included:

1.  **Data Preprocessing:** Combined 'title' and 'text' columns, followed by extensive cleaning (lowercasing, punctuation/stopwords removal, stemming).
2.  **Model Training:** Used `TfidfVectorizer` for text numericalization and trained a `PassiveAggressiveClassifier`.
3.  **Evaluation:** The initial model achieved approximately **93.13% accuracy**.

In summary, the model effectively identifies fake news, demonstrating strong performance with standard NLP techniques. Further improvements could involve advanced NLP methods or larger datasets.